In [1]:
import os
import site

import os
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

In [ ]:
companies = pd.read_parquet('stocknet-dataset/stock_table.parquet')
tweets = pd.read_parquet('stocknet-dataset/stock_tweets_sentiment_unifyemotion_stanceScore_nomerge.parquet')
stocks = pd.read_parquet('stocknet-dataset/stock_prices.parquet')

companies = companies.rename(columns={'symbol': 'ticker'})

companies.columns = [x.lower() for x in companies.columns]
tweets.columns = [x.lower() for x in tweets.columns]
stocks.columns = [x.lower() for x in stocks.columns]

tweets['stance_positive'] = (tweets['stance_label'] == 'Positive').astype(int)
tweets['stance_negative'] = (tweets['stance_label'] == 'Negative').astype(int)

tweets_merged = tweets.groupby(['date', 'ticker'], as_index=False).agg({
    'text': lambda x: ' '.join(x),
    'sentiment': lambda x: x.mean(),
    'emotion_anger': 'sum',
    'emotion_disgust': 'sum',
    'emotion_fear': 'sum',
    'emotion_joy': 'sum',
    'emotion_neutral': 'sum',
    'emotion_sadness': 'sum',
    'emotion_surprize': 'sum',
    'stance_positive': 'sum',
    'stance_negative': 'sum'
})


tweets_merged['date'] = pd.to_datetime(tweets_merged['date'])
stocks['date'] = pd.to_datetime(stocks['date'])

"""
master_df = stocks.merge(
    tweets_merged,
    on=["date", "ticker"]
)
"""


master_df = pd.merge(
    stocks,
    tweets_merged,
    on=["date", "ticker"],
    how='left'
)

tweet_feature_cols = ['sentiment', 'emotion_anger', 'emotion_disgust', 'emotion_fear', 'emotion_joy', 'emotion_neutral', 'emotion_sadness', 'emotion_surprize', 'stance_positive', 'stance_negative']
for col in tweet_feature_cols:
    if col in master_df.columns:
        master_df[col].fillna(0, inplace=True)



companies = companies.rename(columns={'symbol': 'ticker'})

master_df = pd.merge(master_df, companies[['ticker', 'sector', 'company']], on='ticker', how='left')

master_df = master_df.rename(columns={'close': 'close_price', 'company': 'company_name'})


print(f"Shape of master_df before dropping NaNs: {master_df.shape}")
#master_df.dropna(inplace=True)
print(f"Shape of master_df after dropping NaNs: {master_df.shape}")

master_df.rename(columns={'close_price': 'close'}, inplace=True)


master_df.sort_values(by=['ticker', 'date'], inplace=True)


def apply_ta_indicators(df_group):
    df_group.set_index(pd.DatetimeIndex(df_group['date']), inplace=True)
    #Trend
    df_group.ta.ema(length=12, append=True)
    df_group.ta.ema(length=26, append=True)
    df_group.ta.ema(length=50, append=True)

    df_group.ta.macd(fast=12, slow=26, signal=9, append=True)



    df_group.ta.rsi(length=14, append=True)
    df_group.ta.stochrsi(length=14, append=True)


    df_group.ta.atr(length=14, append=True)

    bb = ta.bbands(df_group['close'], length=20, std=2)
    df_group['BB_upper'] = bb['BBU_20_2.0']
    df_group['BB_middle'] = bb['BBM_20_2.0']
    df_group['BB_lower'] = bb['BBL_20_2.0']


    df_group.ta.obv(append=True)
    return df_group.reset_index(drop=True)

master_df = master_df.groupby('ticker').apply(apply_ta_indicators)


Shape of master_df before dropping NaNs: (108592, 21)
Shape of master_df after dropping NaNs: (108592, 21)


In [4]:
master_df


date       open       high        low      close  adj close  \
ticker                                                                          
AAPL   0    2012-09-04  95.108574  96.448570  94.928574  96.424286  87.121140   
       1    2012-09-05  96.510002  96.621429  95.657143  95.747147  86.509338   
       2    2012-09-06  96.167145  96.898575  95.828575  96.610001  87.288956   
       3    2012-09-07  96.864288  97.497147  96.538574  97.205711  87.827171   
       4    2012-09-10  97.207146  97.612854  94.585716  94.677139  85.542564   
...                ...        ...        ...        ...        ...        ...   
XOM    1253 2017-08-28  76.900002  76.940002  76.260002  76.470001  76.470001   
       1254 2017-08-29  76.209999  76.489998  76.080002  76.449997  76.449997   
       1255 2017-08-30  76.239998  76.449997  76.059998  76.099998  76.099998   
       1256 2017-08-31  76.269997  76.489998  76.050003  76.330002  76.330002   
       1257 2017-09-01  76.370003  76.849998  76.320000  76.570000  76.570000   

                  volume ticker text  sentiment  ...  MACDh_12_26_9  \
ticker                                           ...                  
AAPL   0      91973000.0   AAPL  NaN        0.0  ...            NaN   
       1      84093800.0   AAPL  NaN        0.0  ...            NaN   
       2      97799100.0   AAPL  NaN        0.0  ...            NaN   
       3      82416600.0   AAPL  NaN        0.0  ...            NaN   
       4     121999500.0   AAPL  NaN        0.0  ...            NaN   
...                  ...    ...  ...        ...  ...            ...   
XOM    1253    8229700.0    XOM  NaN        0.0  ...      -0.107548   
       1254    7060400.0    XOM  NaN        0.0  ...      -0.069077   
       1255    8218000.0    XOM  NaN        0.0  ...      -0.054652   
       1256   15641700.0    XOM  NaN        0.0  ...      -0.018917   
       1257    7340800.0    XOM  NaN        0.0  ...       0.028984   

             MACDs_12_26_9     RSI_14  STOCHRSIk_14_14_3_3  \
ticker                                                       
AAPL   0               NaN        NaN                  NaN   
       1               NaN        NaN                  NaN   
       2               NaN        NaN                  NaN   
       3               NaN        NaN                  NaN   
       4               NaN        NaN                  NaN   
...                    ...        ...                  ...   
XOM    1253      -0.972858  31.975492            35.117121   
       1254      -0.990127  31.851847            48.597552   
       1255      -1.003790  29.688704            55.025431   
       1256      -1.008519  32.913052            73.940933   
       1257      -1.001273  36.200731            85.986580   

             STOCHRSId_14_14_3_3   ATRr_14   BB_upper  BB_middle   BB_lower  \
ticker                                                                        
AAPL   0                     NaN       NaN        NaN        NaN        NaN   
       1                     NaN       NaN        NaN        NaN        NaN   
       2                     NaN       NaN        NaN        NaN        NaN   
       3                     NaN       NaN        NaN        NaN        NaN   
       4                     NaN       NaN        NaN        NaN        NaN   
...                          ...       ...        ...        ...        ...   
XOM    1253            31.775404  0.786087  81.525829    78.2435  74.961171   
       1254            38.712818  0.759224  81.303475    78.0575  74.811525   
       1255            46.246701  0.732850  80.964170    77.8325  74.700830   
       1256            59.187972  0.711932  80.569554    77.6245  74.679446   
       1257            71.650981  0.698937  80.167620    77.4425  74.717380   

                     OBV  
ticker                    
AAPL   0      91973000.0  
       1       7879200.0  
       2     105678300.0  
       3     188094900.0  
       4      66095400.0  
...             

In [5]:
columns_to_check = ['EMA_12', 'EMA_26','EMA_50','MACD_12_26_9','MACDh_12_26_9','MACDs_12_26_9','RSI_14','ATRr_14','STOCHRSIk_14_14_3_3','STOCHRSId_14_14_3_3','ATRr_14','BB_upper','BB_middle','BB_lower','OBV']
master_df = master_df.dropna(subset=columns_to_check)

In [6]:
master_df = master_df.reset_index(drop=True)

# 1-day return (t-1 -> t)
master_df['ret_1d'] = master_df.groupby('ticker')['close'].pct_change()
# 1-day rolling return is the same as ret_1d; keep a named column for clarity
master_df['roll_ret_1d'] = master_df['ret_1d']
# 5-day rolling mean of past 1-day returns
master_df['roll_ret_5d'] = (
    master_df.groupby('ticker')['ret_1d']
    .transform(lambda s: s.rolling(5, min_periods=1).mean())
)
# 20-day rolling mean of past 1-day returns
master_df['roll_ret_20d'] = (
    master_df.groupby('ticker')['ret_1d']
    .transform(lambda s: s.rolling(20, min_periods=1).mean())
)


In [7]:
master_df.reset_index(drop=True, inplace=True)
master_df

,date,open,high,low,close,adj close,volume,ticker,text,sentiment,...,STOCHRSId_14_14_3_3,ATRr_14,BB_upper,BB_middle,BB_lower,OBV,ret_1d,roll_ret_1d,roll_ret_5d,roll_ret_20d
0,2012-11-14,77.928574,78.207146,76.597145,76.697144,69.613815,119292600.0,AAPL,NaN,0.0,...,19.582354,2.377852,94.648550,84.401357,74.154164,-1.014356e+09,NaN,NaN,NaN,NaN
1,2012-11-15,76.790001,77.071426,74.660004,75.088570,68.153778,197477700.0,AAPL,NaN,0.0,...,19.993462,2.380310,93.761634,83.514428,73.267223,-1.211834e+09,-0.020973,-0.020973,-0.020973,-0.020973
2,2012-11-16,75.028572,75.714287,72.250000,75.382858,68.420891,316723400.0,AAPL,NaN,0.0,...,16.363641,2.459547,92.716200,82.679214,72.642228,-8.951103e+08,0.003919,0.003919,-0.008527,-0.008527
3,2012-11-19,77.244286,81.071426,77.125717,80.818573,73.354591,205829400.0,AAPL,NaN,0.0,...,22.576123,2.695187,91.617665,82.201285,72.784906,-6.892809e+08,0.072108,0.072108,0.018351,0.018351
4,2012-11-20,81.701431,81.707146,79.225716,80.129997,72.729614,160688500.0,AAPL,NaN,0.0,...,40.436207,2.679612,91.027780,81.851785,72.675790,-8.499694e+08,-0.008520,-0.008520,0.011634,0.011634
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104215,2017-08-28,76.900002,76.940002,76.260002,76.470001,76.470001,8229700.0,XOM,NaN,0.0,...,31.775404,0.786087,81.525829,78.243500,74.961171,-2.688251e+08,-0.003259,-0.003259,0.000243,-0.002261
104216,2017-08-29,76.209999,76.489998,76.080002,76.449997,76.449997,7060400.0,XOM,NaN,0.0,...,38.712818,0.759224,81.303475,78.057500,74.811525,-2.758855e+08,-0.000262,-0.000262,-0.000752,-0.002355
104217,2017-08-30,76.239998,76.449997,76.059998,76.099998,76.099998,8218000.0,XOM,NaN,0.0,...,46.246701,0.732850,80.964170,77.832500,74.700830,-2.841035e+08,-0.004578,-0.004578,-0.001329,-0.002852
104218,2017-08-31,76.269997,76.489998,76.050003,76.330002,76.330002,15641700.0,XOM,NaN,0.0,...,59.187972,0.711932,80.569554,77.624500,74.679446,-2.684618e+08,0.003022,0.003022,0.000007,-0.002633


In [8]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104220 entries, 0 to 104219
Data columns (total 39 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   date                 104220 non-null  datetime64[ns]
 1   open                 104220 non-null  float64       
 2   high                 104220 non-null  float64       
 3   low                  104220 non-null  float64       
 4   close                104220 non-null  float64       
 5   adj close            104220 non-null  float64       
 6   volume               104220 non-null  float64       
 7   ticker               104220 non-null  object        
 8   text                 19297 non-null   object        
 9   sentiment            104220 non-null  float64       
 10  emotion_anger        104220 non-null  float64       
 11  emotion_disgust      104220 non-null  float64       
 12  emotion_fear         104220 non-null  float64       
 13  emotion_joy   

In [ ]:
master_df.to_parquet('stocknet-dataset/stock_price_rawemotion_stance.parquet', index=False)